## Импорты

In [1]:
import os
import datetime

import warnings
warnings.filterwarnings('ignore')

import IPython
import IPython.display
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import plotly.graph_objects as go
from plotly.subplots import make_subplots

mpl.rcParams['figure.figsize'] = (12, 6)
mpl.rcParams['axes.grid'] = False

import talib
import backtesting
import yfinance

from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import VotingClassifier

from catboost import CatBoostClassifier

Loading BokehJS ...

## Исходные данные

In [2]:
# Данные по индексу
# https://www.moex.com/ru/index/MOEXBC/archive?from=2009-04-24&till=2024-11-25&sort=TRADEDATE&order=desc

# Курс доллара
# https://investfunds.ru/indexes/39/

!wget 'https://drive.google.com/uc?export=download&id=1QBemvMmFNhiR25JPr_tO-4bYh-1NV0zD' -O 'moexbc.csv'
!wget 'https://drive.google.com/uc?export=download&id=1UYbOHL95Xe0MOEY7VOaTn17sto8MWl3B' -O 'usd_rub-(банк-россии).xlsx'

--2025-08-26 21:06:35--  https://drive.google.com/uc?export=download&id=1QBemvMmFNhiR25JPr_tO-4bYh-1NV0zD
Resolving drive.google.com (drive.google.com)... 173.194.205.113, 173.194.205.138, 173.194.205.139, ...
Connecting to drive.google.com (drive.google.com)|173.194.205.113|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1QBemvMmFNhiR25JPr_tO-4bYh-1NV0zD&export=download [following]
--2025-08-26 21:06:35--  https://drive.usercontent.google.com/download?id=1QBemvMmFNhiR25JPr_tO-4bYh-1NV0zD&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.250.187.129
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.250.187.129|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 648088 (633K) [application/octet-stream]
Saving to: ‘moexbc.csv’

moexbc.csv          100%[===================>] 632,90K  2,12MB/s    in 0,3s    

2025-0

In [3]:
# Индекс
db = pd.read_csv('moexbc.csv', encoding = 'windows-1251', sep = ';')
db = db.reset_index()
clns = list(db.loc[0, :])
db = db.iloc[1:, :].reset_index(drop=True)
db.columns = clns
db = db[['TRADEDATE','CLOSE','OPEN','HIGH','LOW','VALUE']]
db.columns = ['date', 'close','open', 'high', 'low', 'volume']
for c in ['close','open', 'high', 'low', 'volume']:
    db[c] = db[c].apply(lambda x: float(str(x).replace(',', '.')))
db['date'] = db['date'].apply(lambda x: str(x)[:10])

# Курс валюты
s0 = 'usd_rub-(банк-россии).xlsx'
d0 = pd.read_excel(s0)
d0.columns = ['date', 'USD_RUB']
d0['date'] = d0['date'].apply(lambda x: str(x)[:10])
d0['date'] = d0['date'].apply(lambda x: x[8:10]+'.'+x[5:7]+'.'+x[:4])
d0 = d0[['date', 'USD_RUB']]

# Собираем вместе
db = db.merge(d0, on='date', how='left')
db['close_RUR'] = db['close']
db['close'] = db['close']/db['USD_RUB']


# Обработка даты
c= 'date'
db[c] = db[c].apply(lambda x: x.split('.')[2]+'-'+x.split('.')[1]+'-'+x.split('.')[0])
db[c] = db[c].apply(lambda x: str(x))

c= 'date'
db[c] = db[c].apply(lambda x: str(x))
db = db.interpolate(method='linear').fillna(method='ffill').fillna(method='bfill')
db[c] = db[c].apply(pd.to_datetime, errors='coerce')
db = db.sort_values(c, ascending=True).reset_index(drop=True)

# Перевод всех значений в числовой формат
for c in [x for x in db.columns if x != 'date']:
    db[c] = db[c].apply(pd.to_numeric, errors='coerce')

db

,date,close,open,high,low,volume,USD_RUB,close_RUR
0,2009-04-24,185.766972,6285.76,6362.62,6239.72,8.292833e+08,33.7848,6276.10
1,2009-04-27,181.275454,6276.10,6276.10,6033.84,1.101359e+09,33.4187,6057.99
2,2009-04-28,176.690007,6057.99,6116.65,5885.04,1.978925e+09,33.3904,5899.75
3,2009-04-29,182.067040,5899.75,6111.45,5899.75,2.348292e+09,33.5533,6108.95
4,2009-04-30,188.417130,6108.95,6353.44,6108.95,1.739327e+09,33.2491,6264.70
...,...,...,...,...,...,...,...,...
3894,2024-11-06,174.055185,17171.89,17394.37,16992.03,8.876278e+10,98.0562,17067.19
3895,2024-11-07,175.887363,17067.19,17301.28,16902.70,4.434799e+10,98.2236,17276.29
3896,2024-11-08,178.794281,17488.76,17660.77,17420.44,6.127687e+10,98.0726,17534.82
3897,2024-11-11,182.055329,17839.05,17926.57,17712.09,7.750142e+10,97.8335,17811.11


## Фильтруем данные
классический метод IQR-фильтрации (InterQuartile Range)

В scikit-learn нет готового трансформера “удалить выбросы по IQR”, поэтому напишем кастомный класс-трансформер и упакуем его в sklearn-совместимый трансформер (через BaseEstimator, TransformerMixin), чтобы встроить в Pipeline.

In [4]:
class IQRCarryForwardOutlierRemover(BaseEstimator, TransformerMixin):
    """
    Заменяет выбросы (по правилу IQR) на предыдущее корректное значение.
    - Выбросы: x < Q1 - factor*IQR или x > Q3 + factor*IQR
    - По умолчанию обрабатывает все числовые столбцы, кроме date-колонки.
    - Сохраняет DataFrame и порядок столбцов.
    """
    def __init__(self, factor=1.5, date_col='date', columns=None, sort_by_date=True):
        self.factor = float(factor)
        self.date_col = date_col
        self.columns = columns  # None => авто-выбор числовых
        self.sort_by_date = sort_by_date

    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Ожидаю pandas.DataFrame на входе.")
        df = X.copy()

        # авто-выбор колонок: все числовые, кроме date
        if self.columns is None:
            numeric_cols = df.select_dtypes(include=[np.number, "float", "int"]).columns.tolist()
            self.columns_ = [c for c in numeric_cols if c != self.date_col]
        else:
            self.columns_ = list(self.columns)

        # посчитаем пороги по каждому столбцу
        self.bounds_ = {}
        for col in self.columns_:
            s = pd.to_numeric(df[col], errors="coerce")
            q1 = np.nanpercentile(s, 25)
            q3 = np.nanpercentile(s, 75)
            iqr = q3 - q1
            lower = q1 - self.factor * iqr
            upper = q3 + self.factor * iqr
            self.bounds_[col] = (lower, upper)

        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Ожидаю pandas.DataFrame на входе.")
        if not hasattr(self, "bounds_"):
            raise RuntimeError("Сначала вызовите fit().")

        df = X.copy()

        # сортировка по дате (если нужно) — важна для "последнего корректного"
        if self.sort_by_date and self.date_col in df.columns:
            df = df.sort_values(self.date_col).reset_index(drop=True)

        for col in self.columns_:
            s = pd.to_numeric(df[col], errors="coerce")

            lower, upper = self.bounds_[col]
            is_outlier = (s < lower) | (s > upper)

            # выбросы -> NaN, затем тянем последнее корректное значение вперёд
            s_masked = s.mask(is_outlier)
            s_filled = s_masked.ffill()

            # если выбросы в самом начале (нет "прошлого" значения) — оставляем исходные
            leading = s_filled.isna()
            if leading.any():
                s_filled[leading] = s[leading]

            df[col] = s_filled

        return df

In [5]:

pipe_filter = Pipeline([
    ("iqr_cf", IQRCarryForwardOutlierRemover(
        factor=1.5,
        date_col='date',
        columns=['close','open','high','low','volume','USD_RUB','close_RUR'],  # можно не указывать: выберет числовые сам
        sort_by_date=True
    ))
])

df_clean = pipe_filter.fit_transform(db)  # db — ваш исходный DataFrame

## Создание признаков

In [6]:
# RSI

def calculate_rsi(new_data: pd.DataFrame, column='close', window=14)->pd.Series:
    delta = new_data[column].diff()
    gain = (delta.where(delta > 0, 0)).fillna(0)
    loss = (-delta.where(delta < 0, 0)).fillna(0)
    
    avg_gain = gain.rolling(window=window, min_periods=1).mean()
    avg_loss = loss.rolling(window=window, min_periods=1).mean()
    
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    rsi.fillna(0, inplace=True)
    return rsi

In [7]:
# Moving RSI
def calculate_moving_rsi(rsi: pd.Series, window=14):
    return rsi.rolling(window=window, min_periods=1).mean()

In [8]:
class TechIndicatorsTransformer(BaseEstimator, TransformerMixin):
    """Добавляет новые фичи:
    - RSI и его скользящее среднне
    - MACD, MACD_Signal, MACD_Hist
    - SMA короткое идлинное
    Args:
        BaseEstimator (_type_): _description_
        TransformerMixin (_type_): _description_
    """
    def __init__(
        self,
        price_col="close",
        rsi_window=14,
        rsi_ma_window=14,
        sma_short=20,
        sma_long=50,
        date_col="date",
        sort_by_date=True,
    ):
        self.price_col = price_col
        self.rsi_window = rsi_window
        self.rsi_ma_window = rsi_ma_window
        self.sma_short = sma_short
        self.sma_long = sma_long
        self.date_col = date_col
        self.sort_by_date = sort_by_date

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("TechnicalIndicatorsTransformer ожидает pandas.DataFrame")

        df = X.copy()

        # сортировка по дате (для временного ряда)
        if self.sort_by_date and self.date_col in df.columns:
            df = df.sort_values(self.date_col).reset_index(drop=True)

        price = df[self.price_col].astype(float).values

        # RSI
        rsi = calculate_rsi(df, column=self.price_col, window=self.rsi_window)
        df[f"RSI_{self.rsi_window}"] = rsi
        df[f"RSI_{self.rsi_window}_MA{self.rsi_ma_window}"] = calculate_moving_rsi(
            rsi, window=self.rsi_ma_window
        )

        # MACD
        macd, macd_signal, macd_hist = talib.MACD(
            price, fastperiod=12, slowperiod=26, signalperiod=9
        )
        df["MACD"] = macd
        df["MACD_Signal"] = macd_signal
        df["MACD_Hist"] = macd_hist
        
        # Рассчитываем TripleEMA и MACD
        df['tema'] = talib.TEMA(df['close'], timeperiod=24)
        df['macd'], df['macd_signal'], df['macd_hist'] = talib.MACD(df['close'], fastperiod=12, slowperiod=26, signalperiod=9)

        # SMA short/long
        df[f"SMA_{self.sma_short}"] = talib.SMA(price, timeperiod=self.sma_short)
        df[f"SMA_{self.sma_long}"] = talib.SMA(price, timeperiod=self.sma_long)

        df.fillna(0, inplace=True)
        
        return df

In [9]:
pipe_transform = Pipeline([
    ("ti", TechIndicatorsTransformer(
        price_col="close",
        rsi_window=14,
        rsi_ma_window=14,
        sma_short=20,
        sma_long=50,
    ))
])

df_new = pipe_transform.fit_transform(df_clean)  # db = DataFrame с колонками ['date','close','open','high','low',...]
df_new.head()

,date,close,open,high,low,volume,USD_RUB,close_RUR,RSI_14,RSI_14_MA14,MACD,MACD_Signal,MACD_Hist,tema,macd,macd_signal,macd_hist,SMA_20,SMA_50
0,2009-04-24,185.766972,6285.76,6362.62,6239.72,8.292833e+08,33.7848,6276.10,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2009-04-27,181.275454,6276.10,6276.10,6033.84,1.101359e+09,33.4187,6057.99,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2009-04-28,176.690007,6057.99,6116.65,5885.04,1.978925e+09,33.3904,5899.75,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2009-04-29,182.067040,5899.75,6111.45,5899.75,2.348292e+09,33.5533,6108.95,37.201007,9.300252,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2009-04-30,188.417130,6108.95,6353.44,6108.95,1.739327e+09,33.2491,6264.70,56.369320,18.714065,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Модель на базе тех анализа

In [36]:
def _ema(s: pd.Series, period: int) -> pd.Series:
    return pd.Series(s, index=s.index, dtype=float).ewm(span=period, adjust=False).mean()

def _tema(close: pd.Series, period: int) -> pd.Series:
    # TEMA = 3*EMA1 - 3*EMA2 + EMA3
    ema1 = _ema(close, period)
    ema2 = _ema(ema1, period)
    ema3 = _ema(ema2, period)
    return 3*ema1 - 3*ema2 + ema3

def _macd_from_close(close: pd.Series, fast=12, slow=26, signal=9):
    ema_fast = _ema(close, fast)
    ema_slow = _ema(close, slow)
    macd = ema_fast - ema_slow
    macd_signal = _ema(macd, signal)
    return macd, macd_signal

class RuleSignalClassifier(BaseEstimator, ClassifierMixin):
    """
    Классификатор по правилу:
      buy  (1):  macd > macd_signal  AND  close > tema
      sell (-1): macd < macd_signal  AND  close < tema
      hold (0):  иначе

    Параметры:
      close_col: имя колонки цены закрытия
      macd_col, macd_signal_col, tema_col: имена колонок с индикаторами, если уже есть
      compute_if_missing: считать индикаторы из close, если колонок нет
      macd_fast, macd_slow, macd_signal: параметры MACD, если считаем
      tema_period: период TEMA, если считаем
      neutral_class: метка «нет сигнала» (по умолчанию 0)
    """

    
    def __init__(self,
                 close_col="close",
                 macd_col="macd",
                 macd_signal_col="macd_signal",
                 tema_col="tema",
                 compute_if_missing=True,
                 macd_fast=12, macd_slow=26, macd_signal=9,
                 tema_period=30,
                 neutral_class=0):
        self.close_col = close_col
        self.macd_col = macd_col
        self.macd_signal_col = macd_signal_col
        self.tema_col = tema_col
        self.compute_if_missing = compute_if_missing
        self.macd_fast = macd_fast
        self.macd_slow = macd_slow
        self.macd_signal = macd_signal
        self.tema_period = tema_period
        self.neutral_class = neutral_class

    # обучаться тут нечему — но sklearn ожидает fit
    def fit(self, X: pd.DataFrame, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Ожидается pandas.DataFrame")
        # объявим порядок классов для predict_proba
        self.classes_ = np.array([-1, self.neutral_class, 1], dtype=int)
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        df = self._ensure_indicators(X)
        macd = pd.to_numeric(df[self.macd_col], errors="coerce")
        macd_signal = pd.to_numeric(df[self.macd_signal_col], errors="coerce")
        close = pd.to_numeric(df[self.close_col], errors="coerce")
        tema = pd.to_numeric(df[self.tema_col], errors="coerce")

        # условия
        buy = (macd > macd_signal) & (close > tema)
        sell = (macd < macd_signal) & (close < tema)

        out = np.full(len(df), self.neutral_class, dtype=int)
        out[sell.fillna(False).values] = -1
        out[buy.fillna(False).values] = 1
        return out

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        """
        Жёсткие «вероятности»: 1.0 для предсказанного класса, 0.0 для остальных.
        Нужно лишь для совместимости со scorer’ами, если потребуется.
        """
        y = self.predict(X)
        proba = np.zeros((len(y), 3), dtype=float)  # порядок: [-1, neutral, +1]
        for i, label in enumerate(y):
            if label == -1:
                proba[i, 0] = 1.0
            elif label == 1:
                proba[i, 2] = 1.0
            else:
                proba[i, 1] = 1.0
        return proba

    # --- вспомогательное ---
    def _ensure_indicators(self, X: pd.DataFrame) -> pd.DataFrame:
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Ожидается pandas.DataFrame")
        if self.close_col not in X.columns:
            raise KeyError(f"Не найдена колонка '{self.close_col}'")

        df = X.copy()

        # MACD
        need_macd = (self.macd_col not in df.columns) or (self.macd_signal_col not in df.columns)
        if need_macd:
            if not self.compute_if_missing:
                missing = [c for c in [self.macd_col, self.macd_signal_col] if c not in df.columns]
                raise KeyError(f"Отсутствуют {missing}, а вычислять запрещено (compute_if_missing=False)")
            macd, macd_sig = _macd_from_close(
                df[self.close_col].astype(float),
                fast=self.macd_fast, slow=self.macd_slow, signal=self.macd_signal
            )
            df[self.macd_col] = macd
            df[self.macd_signal_col] = macd_sig

        # TEMA
        if self.tema_col not in df.columns:
            if not self.compute_if_missing:
                raise KeyError(f"Отсутствует '{self.tema_col}', а вычислять запрещено (compute_if_missing=False)")
            df[self.tema_col] = _tema(df[self.close_col].astype(float), self.tema_period)

        return df

In [37]:
rule_clf = RuleSignalClassifier(
    close_col="close",
    macd_col="macd",
    macd_signal_col="macd_signal",
    tema_col="tema",
    compute_if_missing=True,  # посчитает индикаторы из close при отсутствии
    macd_fast=12, macd_slow=26, macd_signal=9,
    tema_period=30,
    neutral_class=0
)

In [38]:
# Итоговый пайплайн
pipe = Pipeline([
    ('filter', pipe_filter),
    ('transformer', pipe_transform),
    ("rule_model", rule_clf),
])

pipe

Pipeline(steps=[('filter',
                 Pipeline(steps=[('iqr_cf',
                                  IQRCarryForwardOutlierRemover(columns=['close',
                                                                         'open',
                                                                         'high',
                                                                         'low',
                                                                         'volume',
                                                                         'USD_RUB',
                                                                         'close_RUR']))])),
                ('transformer',
                 Pipeline(steps=[('ti', TechIndicatorsTransformer())])),
                ('rule_model', RuleSignalClassifier())])

In [39]:
pipe.fit(db)   # db — ваш DataFrame
signals = pipe.predict(db)

In [40]:
pipe

Pipeline(steps=[('filter',
                 Pipeline(steps=[('iqr_cf',
                                  IQRCarryForwardOutlierRemover(columns=['close',
                                                                         'open',
                                                                         'high',
                                                                         'low',
                                                                         'volume',
                                                                         'USD_RUB',
                                                                         'close_RUR']))])),
                ('transformer',
                 Pipeline(steps=[('ti', TechIndicatorsTransformer())])),
                ('rule_model', RuleSignalClassifier())])

## Добавим еще один классификатор в модель

In [41]:
# ---------- DataFrame-обёртка над MinMaxScaler (сохраняем DataFrame-формат) ----------
class DataFrameMinMaxScaler(BaseEstimator, TransformerMixin):
    """
    Масштабирует указанные столбцы DataFrame с сохранением формата DataFrame.
    По умолчанию: скейлит все числовые колонки, КРОМЕ price_col и date_col.
    """
    def __init__(self, columns=None, price_col="close", date_col="date", feature_range=(0,1), clip=False):
        self.columns = columns
        self.price_col = price_col
        self.date_col = date_col
        self.feature_range = feature_range
        self.clip = clip

    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Ожидается pandas.DataFrame")
        df = X

        if self.columns is None:
            numeric = df.select_dtypes(include=[np.number, "float", "int"]).columns.tolist()
            # исключим цену и дату по умолчанию
            self.columns_ = [c for c in numeric if c not in {self.price_col, self.date_col}]
        else:
            self.columns_ = list(self.columns)

        self.scaler_ = MinMaxScaler(feature_range=self.feature_range, clip=self.clip)
        self.scaler_.fit(df[self.columns_].values)
        return self

    def transform(self, X):
        df = X.copy()
        arr = self.scaler_.transform(df[self.columns_].values)
        df.loc[:, self.columns_] = arr
        return df

In [42]:
class CatBoostSignalClassifier(BaseEstimator, ClassifierMixin):
    """
    y[t] =  1, если close[t+1] > close[t] + const_margin
           -1, если close[t+1] < close[t] - const_margin
            0, иначе
    Обучается CatBoost на числовых фичах X (можно уже отмасштабированных шагом до него).
    """
    
    def __init__(self,
                 price_col="close",
                 const_margin=0.0,
                 feature_cols=None,      # если None — все числовые, кроме date
                 date_col="date",
                 sort_by_date=True,
                 cat_params=None,
                 neutral_class=0):
        self.price_col = price_col
        self.const_margin = float(const_margin)
        self.feature_cols = feature_cols
        self.date_col = date_col
        self.sort_by_date = sort_by_date
        self.cat_params = cat_params or {}
        self.neutral_class = neutral_class

    def _build_target(self, close: pd.Series) -> pd.Series:
        nxt = close.shift(-1)
        diff = nxt - close
        y = pd.Series(self.neutral_class, index=close.index, dtype=int)
        y[diff >  self.const_margin] = 1
        y[diff < -self.const_margin] = -1
        return y  # последний индекс останется neutral (нет next)

    def _select_feature_cols(self, df: pd.DataFrame):
        if self.feature_cols is not None:
            return list(self.feature_cols)
        cols = df.select_dtypes(include=[np.number, "float", "int"]).columns.tolist()
        self.feature_cols_ = [c for c in cols if c != self.date_col]  # close остаётся как фича — обычно это ок
        return self.feature_cols_

    def fit(self, X: pd.DataFrame, y=None):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Ожидается pandas.DataFrame")
        df = X.copy()
        if self.sort_by_date and self.date_col in df.columns:
            df = df.sort_values(self.date_col).reset_index(drop=True)

        if self.price_col not in df.columns:
            raise KeyError(f"Нет '{self.price_col}' для построения цели")

        # целевая метка из НЕскейленного close (мы его не трогаем в скейлере)
        y_full = self._build_target(pd.to_numeric(df[self.price_col], errors="coerce"))

        # фичи
        feat_cols = self._select_feature_cols(df)
        X_full = df[feat_cols].copy()

        mask = X_full.notna().all(axis=1) & y_full.notna()
        X_train = X_full[mask]
        y_train = y_full[mask]

        # маппинг классов {-1,0,1} -> {0,1,2}
        self.classes_ = np.array([-1, self.neutral_class, 1], dtype=int)
        label_to_idx = {-1:0, self.neutral_class:1, 1:2}
        y_idx = y_train.map(label_to_idx).astype(int).values

        params = dict(
            loss_function="MultiClass",
            iterations=500,
            depth=6,
            learning_rate=0.05,
            random_seed=42,
            verbose=False
        )
        params.update(self.cat_params or {})

        self.model_ = CatBoostClassifier(**params)
        self.model_.fit(X_train.values, y_idx)

        self.feature_cols_ = X_train.columns.tolist()
        return self

    def _prepare_features(self, X: pd.DataFrame) -> pd.DataFrame:
        if self.sort_by_date and self.date_col in X.columns:
            X = X.sort_values(self.date_col).reset_index(drop=True)
        feats = X[self.feature_cols_].copy()
        feats = feats.fillna(method="ffill").fillna(method="bfill")
        return feats

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        feats = self._prepare_features(X)
        y_idx_pred = self.model_.predict(feats.values).ravel().astype(int)
        idx_to_label = {0:-1, 1:self.neutral_class, 2:1}
        return np.vectorize(idx_to_label.get)(y_idx_pred)

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        feats = self._prepare_features(X)
        return self.model_.predict_proba(feats.values)

In [43]:
# --- трансформеры ---
tech = TechIndicatorsTransformer(
    price_col="close",
    rsi_window=14, rsi_ma_window=14,
    sma_short=20, sma_long=50
)

scaler = DataFrameMinMaxScaler(
    columns=None,        # авто: все числовые, кроме 'close' и 'date'
    price_col="close",
    date_col="date"
)

# --- классификаторы ---
rule = RuleSignalClassifier(
    close_col="close",
    macd_col="macd",
    macd_signal_col="macd_signal",
    tema_col="tema",
    compute_if_missing=True,
    tema_period=30,
    neutral_class=0
)

cat = CatBoostSignalClassifier(
    price_col="close",
    const_margin=0.001,
    feature_cols=None,
    date_col="date",
    sort_by_date=True,
    cat_params=dict(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        random_seed=42,
        verbose=False,
        loss_function="MultiClass",
    ),
    neutral_class=0
)

# --- пайплайны для каждой ветви ---
pipe_cat = Pipeline([
    ("tech", tech),
    ("scaler", scaler),
    ("cat", cat)
])


pipe_rule = Pipeline([
    ("tech", tech),
    ("rule", rule)
])


# --- финальный VotingClassifier ---
pipe = VotingClassifier(
    estimators=[
        ("cat_branch", pipe_cat),
        ("rule_branch", pipe_rule)
    ],
    voting="soft",         # усредняем вероятности
    weights=[0.7, 0.3]     # можно регулировать
)


pipe

VotingClassifier(estimators=[('cat_branch',
                              Pipeline(steps=[('tech',
                                               TechIndicatorsTransformer()),
                                              ('scaler',
                                               DataFrameMinMaxScaler()),
                                              ('cat',
                                               CatBoostSignalClassifier(cat_params={'depth': 6,
                                                                                    'iterations': 500,
                                                                                    'learning_rate': 0.05,
                                                                                    'loss_function': 'MultiClass',
                                                                                    'random_seed': 42,
                                                                                    'verbose': False},
                                                                        const_margin=0.001))])),
                             ('rule_branch',
                              Pipeline(steps=[('tech',
                                               TechIndicatorsTransformer()),
                                              ('rule',
                                               RuleSignalClassifier())]))],
                 voting='soft', weights=[0.7, 0.3])

In [55]:
# Проблема 1 - VoitingClassifier требует y

pipe.fit(db)

TypeError: VotingClassifier.fit() missing 1 required positional argument: 'y'

Это из-за VotingClassifier: он — supervised и требует `y` при `fit`. При вызове `pipe.fit(db)` без целевого вектора он падает с ValueError: Expected array-like … got None.

У нас целевая метка по правилу `close[t+1] vs close[t] ± const`. Построим её заранее и обучим пайплайн на `X` без последней строки (для последнего дня нет «следующего» значения).

In [45]:
CONST_MARGIN = 0.001  # ваш порог в единицах цены (или задайте свой)

def build_target(df: pd.DataFrame, price_col="close", const_margin=CONST_MARGIN) -> pd.Series:
    nxt = df[price_col].shift(-1)
    diff = nxt - df[price_col]
    y = np.where(diff >  const_margin,  1,
        np.where(diff < -const_margin, -1, 0))
    # Последняя строка не имеет "nxt" → убираем
    return pd.Series(y[:-1], index=df.index[:-1], dtype=int)

# 1) Формируем y
y = build_target(db, price_col="close", const_margin=CONST_MARGIN)

# 2) Подготовим X под ту же длину
X_train = db.iloc[:-1].copy()

In [56]:
# Проблема 2 - все шаги внутри voitingClassifier должны быть классификаторами

pipe.fit(X_train, y)

ValueError: The estimator Pipeline should be a classifier.

In [ ]:
# Create a custom ensemble class that doesn't rely on VotingClassifier
class CustomEnsembleClassifier(BaseEstimator, ClassifierMixin):
    """
    Custom ensemble classifier that combines rule-based and ML approaches
    """
    
    def __init__(self, cat_pipeline, rule_pipeline, weights=None):
        self.cat_pipeline = cat_pipeline
        self.rule_pipeline = rule_pipeline
        self.weights = weights or [0.7, 0.3]
        
    def fit(self, X, y=None):
        # Fit both pipelines
        self.cat_pipeline.fit(X, y)
        self.rule_pipeline.fit(X, y)
        
        # Set classes
        self.classes_ = np.array([-1, 0, 1])
        
        return self
    
    def predict_proba(self, X):
        # Get probabilities from both models
        cat_proba = self.cat_pipeline.predict_proba(X)
        rule_proba = self.rule_pipeline.predict_proba(X)
        
        # Weighted average
        ensemble_proba = (self.weights[0] * cat_proba + 
                         self.weights[1] * rule_proba)
        
        return ensemble_proba
    
    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes_[np.argmax(proba, axis=1)]

In [52]:
# Create custom ensemble
ensemble = CustomEnsembleClassifier(
    cat_pipeline=pipe_cat,
    rule_pipeline=pipe_rule,
    weights=[0.7, 0.3]
    )

In [53]:
ensemble.fit(X_train, y)

CustomEnsembleClassifier(cat_pipeline=Pipeline(steps=[('tech',
                                                       TechIndicatorsTransformer()),
                                                      ('scaler',
                                                       DataFrameMinMaxScaler()),
                                                      ('cat',
                                                       CatBoostSignalClassifier(cat_params={'depth': 6,
                                                                                            'iterations': 500,
                                                                                            'learning_rate': 0.05,
                                                                                            'loss_function': 'MultiClass',
                                                                                            'random_seed': 42,
                                                                                            'verbose': False},
                                                                                const_margin=0.001))]),
                         rule_pipeline=Pipeline(steps=[('tech',
                                                        TechIndicatorsTransformer()),
                                                       ('rule',
                                                        RuleSignalClassifier())]),
                         weights=[0.7, 0.3])

In [54]:
ensemble.predict(X_train)

array([-1, -1, -1, ...,  1,  1,  1])

## Нейросетевая модель


# LSTM-модель для предсказания сигнала (по `build_target`)

Ниже добавлена простая LSTM‑модель на Keras/TensorFlow, использующая те же цели (‑1, 0, 1), что и функция `build_target`.
Модель обучается на оконных последовательностях признаков (скользящее окно по строкам `X_train`).

> **Примечания:**
> * Код предполагает, что в ноутбуке уже существуют переменные `X_train` (таблица признаков) и `y` (целевая переменная, полученная `build_target`).
> * Окно можно изменять параметром `WINDOW`.
> * Для запуска требуется установленный TensorFlow (`pip install tensorflow`). Если он не установлен, ячейка выведет понятное сообщение.


In [ ]:

# ===== LSTM-секция: обучение и предсказание сигнала =====
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
except Exception as e:
    raise SystemExit(
        "TensorFlow не установлен. Установите его командой `pip install tensorflow` и перезапустите эту ячейку.\n"
        f"Детали: {e}"
    )

import numpy as np
import pandas as pd

# --- Параметры окна и подготовка данных ---
WINDOW = 32  # длина окна последовательности
# Выберем числовые фичи из X_train (исключим индексные/дата-колонки, если присутствуют)
_feature_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
X_num = X_train[_feature_cols].copy()

# y в значениях {-1, 0, 1}. Переведем в индексы классов {0,1,2} для softmax
signal_to_idx = {-1: 0, 0: 1, 1: 2}
idx_to_signal = np.array([-1, 0, 1])

y_idx = pd.Series([signal_to_idx[int(v)] for v in y.values], index=y.index)

# Построим последовательности длины WINDOW
X_list = []
y_list = []
idx_list = []  # чтобы восстановить индексы в исходном DataFrame

for end in range(WINDOW, len(X_num) + 1):
    start = end - WINDOW
    seq = X_num.iloc[start:end].values.astype("float32")
    # y соответствует "последней" позиции в окне (end-1)
    target_idx = y_idx.iloc[end - 1]
    X_list.append(seq)
    y_list.append(target_idx)
    idx_list.append(X_num.index[end - 1])

X_seq = np.stack(X_list, axis=0)  # (samples, WINDOW, features)
y_seq = np.array(y_list, dtype="int64")
idx_seq = pd.Index(idx_list, name=X_num.index.name)

n_features = X_seq.shape[2]

# --- Архитектура LSTM ---
model = keras.Sequential([
    layers.Input(shape=(WINDOW, n_features)),
    layers.LSTM(64),
    layers.Dense(32, activation="relu"),
    layers.Dense(3, activation="softmax")  # 3 класса: {-1, 0, 1} -> {0,1,2}
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# Обучение: делим на трейн/валидацию через validation_split
history = model.fit(
    X_seq, y_seq,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

# --- Предсказания и восстановление сигналов {-1,0,1} ---
proba = model.predict(X_seq, verbose=0)
pred_idx = proba.argmax(axis=1)
pred_signal = idx_to_signal[pred_idx]

# Series с индексами последних точек окон
signal_lstm = pd.Series(pred_signal, index=idx_seq, name="signal_lstm")

# Добавим в общий DataFrame результатов, если он у вас есть; иначе просто оставим Series
try:
    # если у вас есть DataFrame `db`, можно приклеить колонку:
    if "signal_lstm" in X_train.columns:
        pass  # уже есть
    else:
        pass  # оставим как отдельную серию
except Exception:
    pass

print("Готово: переменная `signal_lstm` содержит предсказанные сигналы LSTM выровненные по индексу последних точек окна.")
print("Пример вывода:")
print(signal_lstm.tail(10))



# PyTorch LSTM-модель и интеграция в ансамбль

В этой секции добавляется реализация LSTM на PyTorch, формирующая сигнал `{-1, 0, 1}` по целям, полученным функцией `build_target`.
После обучения результат объединяется с остальными сигналами `signal_*` через простое **голосование по большинству** (при ничьей — `0`).

> Предполагается наличие переменных `X_train` (таблица признаков) и `y` (цели от `build_target`).

In [ ]:

# ===== PyTorch LSTM: обучение, предсказание и ансамбль =====
import numpy as np
import pandas as pd

try:
    import torch
    from torch import nn
    from torch.utils.data import Dataset, DataLoader, random_split
except Exception as e:
    raise SystemExit(
        "PyTorch не установлен. Установите его командой `pip install torch` и перезапустите эту ячейку.\n"
        f"Детали: {e}"
    )

# ---------- Параметры ----------
WINDOW = 32
HIDDEN = 64
EPOCHS = 20
BATCH_SIZE = 128
LR = 1e-3
VAL_RATIO = 0.2
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ---------- Подготовка данных последовательностей ----------
_feature_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
X_num = X_train[_feature_cols].copy()

signal_to_idx = {-1: 0, 0: 1, 1: 2}
idx_to_signal = np.array([-1, 0, 1])

y_idx = pd.Series([signal_to_idx[int(v)] for v in y.values], index=y.index)

X_list, y_list, idx_list = [], [], []
for end in range(WINDOW, len(X_num) + 1):
    start = end - WINDOW
    seq = X_num.iloc[start:end].values.astype("float32")
    target_idx = y_idx.iloc[end - 1]
    X_list.append(seq)
    y_list.append(int(target_idx))
    idx_list.append(X_num.index[end - 1])

X_seq = np.stack(X_list, axis=0)  # (N, WINDOW, F)
y_seq = np.array(y_list, dtype="int64")
idx_seq = pd.Index(idx_list, name=X_num.index.name)
n_features = X_seq.shape[2]

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)  # (N, T, F)
        self.y = torch.from_numpy(y)  # (N,)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, i):
        return self.X[i], self.y[i]

ds = SeqDataset(X_seq, y_seq)

# Разделим на train/val
n_total = len(ds)
n_val = int(n_total * VAL_RATIO)
n_train = n_total - n_val
train_ds, val_ds = random_split(ds, [n_train, n_val], generator=torch.Generator().manual_seed(SEED))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
full_loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)

# ---------- Модель ----------
class LSTMSignal(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=1, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )
    def forward(self, x):
        # x: (B, T, F)
        out, _ = self.lstm(x)
        # берем последнее скрытое состояние по времени:
        last = out[:, -1, :]
        logits = self.head(last)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_torch = LSTMSignal(input_size=n_features, hidden_size=HIDDEN).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_torch.parameters(), lr=LR)

# ---------- Цикл обучения ----------
def evaluate(loader):
    model_torch.eval()
    correct, total, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model_torch(xb)
            loss = criterion(logits, yb)
            loss_sum += loss.item() * yb.size(0)
            pred = logits.argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += yb.size(0)
    return loss_sum / max(1, total), correct / max(1, total)

best_val = float("inf")
for epoch in range(1, EPOCHS + 1):
    model_torch.train()
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = model_torch(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
    val_loss, val_acc = evaluate(val_loader)
    print(f"Epoch {epoch:02d}: val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")
    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.cpu().clone() for k, v in model_torch.state_dict().items()}

# восстановим лучшую модель (по val_loss)
if 'best_state' in locals():
    model_torch.load_state_dict(best_state)

# ---------- Предсказание для всего массива ----------
model_torch.eval()
all_logits = []
with torch.no_grad():
    for xb, _ in full_loader:
        xb = xb.to(device)
        logits = model_torch(xb)
        all_logits.append(logits.cpu())
all_logits = torch.cat(all_logits, dim=0)            # (N, 3)
pred_idx = all_logits.argmax(dim=1).numpy()          # {0,1,2}
pred_signal = idx_to_signal[pred_idx]                # {-1,0,1}

signal_lstm_torch = pd.Series(pred_signal, index=idx_seq, name="signal_lstm_torch")
print("Создана серия `signal_lstm_torch` (PyTorch). Пример:")
print(signal_lstm_torch.tail(10))

# ---------- Интеграция в ансамбль ----------
# Ищем все переменные вида signal_* в глобальном пространстве (Series), выровненные по idx_seq
import inspect, builtins

def _collect_signal_series(globs, index_target):
    series_map = {}
    for name, obj in globs.items():
        if name.startswith("signal_") and name not in {"signal_ensemble"}:
            if isinstance(obj, pd.Series):
                # Совпадает ли индекс хотя бы частично?
                if obj.index.intersection(index_target).size > 0:
                    series_map[name] = obj.astype(int)
    return series_map

_globs = globals()
signals = _collect_signal_series(_globs, idx_seq)

# всегда добавим текущую
signals["signal_lstm_torch"] = signal_lstm_torch

# Собираем в один DataFrame по общему пересечению индексов
if len(signals) >= 2:
    # выравнивание по пересечению индексов всех серий
    common_index = None
    for s in signals.values():
        common_index = s.index if common_index is None else common_index.intersection(s.index)
    # если у кого-то пусто — пропускаем
    signals = {k: v.loc[common_index] for k, v in signals.items() if v.loc[common_index].shape[0] > 0}

signals_df = pd.DataFrame(signals).sort_index()

def majority_vote(row):
    # Голосование по большинству. При ничьей -> 0
    counts = row.value_counts()
    if counts.empty:
        return 0
    top = counts.index[counts.values.argmax()]
    # проверим ничью
    if (counts == counts.max()).sum() > 1:
        return 0
    return int(top)

signal_ensemble = signals_df.apply(majority_vote, axis=1).astype(int)
signal_ensemble.name = "signal_ensemble"

print("\nАнсамбль сформирован из сигналов:", list(signals_df.columns))
print("Размер ансамбля:", len(signal_ensemble))
print("Пример:")
print(signal_ensemble.tail(10))

# По желанию можно сохранить состояние модели
# torch.save(model_torch.state_dict(), "lstm_signal_model.pth")



# PyTorch LSTM‑модель и интеграция в ансамбль

Теперь добавим версию LSTM на **PyTorch** и встроим её в ансамбль с уже существующими моделями.  
Ансамбль будет усреднять предсказанные вероятности классов всех моделей (sklearn, Keras и PyTorch).


In [ ]:

# ===== PyTorch LSTM модель =====
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class SeqDataset(Dataset):
    def __init__(self, X, y, window=32):
        self.X = []
        self.y = []
        self.idx = []
        for end in range(window, len(X) + 1):
            start = end - window
            seq = X.iloc[start:end].values.astype("float32")
            self.X.append(seq)
            self.y.append(y.iloc[end - 1])
            self.idx.append(X.index[end - 1])
        self.X = torch.tensor(self.X)
        self.y = torch.tensor(self.y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class LSTMPredictor(nn.Module):
    def __init__(self, n_features, hidden_size=64, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size, batch_first=True)
        self.fc1 = nn.Linear(hidden_size, 32)
        self.fc2 = nn.Linear(32, num_classes)
        self.relu = nn.ReLU()
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # последний hidden state
        out = self.relu(self.fc1(out))
        out = self.fc2(out)
        return out

# Подготовка данных
WINDOW = 32
_feature_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
X_num = X_train[_feature_cols].copy()
signal_to_idx = {-1:0, 0:1, 1:2}
y_idx = y.map(signal_to_idx)
dataset = SeqDataset(X_num, y_idx, window=WINDOW)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Модель
device = "cuda" if torch.cuda.is_available() else "cpu"
model_torch = LSTMPredictor(n_features=X_num.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_torch.parameters(), lr=1e-3)

# Обучение
for epoch in range(5):  # для примера мало эпох
    model_torch.train()
    total_loss = 0
    for xb, yb in dataloader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model_torch(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss {total_loss/len(dataloader):.4f}")

# Предсказания
model_torch.eval()
with torch.no_grad():
    X_all = torch.tensor(dataset.X).to(device)
    logits = model_torch(X_all)
    proba_torch = torch.softmax(logits, dim=1).cpu().numpy()
    pred_idx = proba_torch.argmax(axis=1)
idx_to_signal = {0:-1, 1:0, 2:1}
signal_torch = pd.Series([idx_to_signal[i] for i in pred_idx], index=dataset.idx, name="signal_torch")

print("Пример предсказаний PyTorch LSTM:")
print(signal_torch.tail())

# ===== Интеграция в ансамбль =====
# Для примера: усредним вероятности Keras и PyTorch LSTM (если доступны) и классических моделей
ensemble_proba = None
components = []

try:
    ensemble_proba = proba.copy()
    components.append("keras")
except Exception:
    pass

try:
    if ensemble_proba is None:
        ensemble_proba = proba_torch.copy()
    else:
        # усреднение вероятностей
        min_len = min(len(ensemble_proba), len(proba_torch))
        ensemble_proba = ensemble_proba[-min_len:] + proba_torch[-min_len:]
        ensemble_proba /= 2
    components.append("torch")
except Exception:
    pass

if ensemble_proba is not None:
    pred_idx = ensemble_proba.argmax(axis=1)
    signal_ensemble = pd.Series([idx_to_signal[i] for i in pred_idx],
                                index=signal_torch.index[-len(pred_idx):],
                                name="signal_ensemble")
    print(f"Ансамбль сформирован из моделей: {components}")
    print(signal_ensemble.tail())
else:
    print("Нет доступных моделей для ансамбля.")
